# Day 4: Enrichment & Dimensionality Reduction
## Data Analysis workshop — IMBB-FORTH

### Objectives

- Understand enrichment analysis through a concrete toy example
- See how it connects to gene expression and GO terms
- Understand what PCA and UMAP do
- Compare PCA and UMAP on single-cell data

---

## 1. Setup

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import hypergeom
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# Base URL for loading data from anywhere
# REPO = "https://raw.githubusercontent.com/cgenomicslab/imbb-data-analysis/main"
REPO = "../"

print("Libraries loaded.")

---

## 2. Enrichment: Colored Balls Example

Imagine a bag with **100 balls** of 5 colors:

| Color | Count |
|-------|-------|
| Purple | 20 |
| Red | 30 |
| Blue | 25 |
| Green | 15 |
| Yellow | 10 |

You grab a **subset of 15 balls** and count: **8 are purple**.

Purple is 20% of the bag, so you would expect about 3 purple in 15 draws. But you got 8.

**Is purple enriched in your subset, or was it just accidental?**

### The concept

The large circle is the **population**. The small circle is our **subset**. Notice how purple is over-represented in the subset.

![Population of balls](../data/enrichment_schema.png "Coloured balls")

### Expected vs Observed

In [ ]:
# The population
population = ['purple'] * 20 + ['red'] * 30 + ['blue'] * 25 + ['green'] * 15 + ['yellow'] * 10

# What we observed in our subset of 15
observed = {'purple': 8, 'red': 3, 'blue': 2, 'green': 1, 'yellow': 1}
subset_size = 15

# How many of each color in the full bag
bag = {'purple': 20, 'red': 30, 'blue': 25, 'green': 15, 'yellow': 10}

# Compare expected vs observed
print("Color", "\t", "Expected", "\t", "Observed", "\t", "Fold enrichment")
print("-" * 56)
for color in bag:
    expected = subset_size * bag[color] / 100
    fold = round(observed[color] / expected, 2)
    print(color, "\t", round(expected, 1), "\t\t", observed[color], "\t\t", fold)

### Is it significant? We'll use a permutation (randomization) test

Randomly grab 15 balls from the bag thousands of times. How often do we get 8 or more purple?

In [ ]:
# Permutation test
np.random.seed(42)
n_permutations = 10000
purple_counts = []

for i in range(n_permutations):
    random_grab = np.random.choice(population, size=subset_size, replace=False)
    purple_counts.append(np.sum(random_grab == 'purple'))

purple_counts = np.array(purple_counts)

# Plot
sns.histplot(purple_counts, bins=range(0, 13), discrete=True, color='steelblue')
plt.axvline(8, color='red', linewidth=2, label='Observed: 8 purple')
plt.xlabel('Purple balls in random subset of 15')
plt.legend()
plt.show()

p_perm = np.mean(purple_counts >= 8)
print("Permutation p-value:", p_perm)

### The hypergeometric test

The [hypergeometric distribution](https://en.wikipedia.org/wiki/Hypergeometric_distribution) calculates the exact probability of drawing a certain number of "special" items from a population **without replacement**.

It takes four numbers:
- **M**: total population (100 balls)
- **n**: special items in population (20 purple)
- **N**: how many we drew (15)
- **k**: special items we observed (8)

This is the standard test used in GO enrichment tools.

In [ ]:
# Hypergeometric test: exact p-value
p_hyper = hypergeom.sf(7, M=100, n=20, N=15)  # P(X >= 8)

print("Permutation p-value:   ", p_perm)
print("Hypergeometric p-value:", round(p_hyper, 4))
print()
print("Both agree: purple is significantly enriched.")

**Try it:** what if you had found 5 purple balls instead of 8? Change the value and re-run. Still significant?

In [ ]:
# Try it here


### 💡 Exercise 4.1: Test All Colors

**Task:** For each color, calculate the fold enrichment and hypergeometric p-value.
Which colors are enriched? Which are depleted?

In [ ]:
# Test all colors
print("Color\t\tObs\tExp\tFold\tP-value\t\tEnriched?")
print("-" * 70)

for color in bag:
    obs = observed[color]
    exp = subset_size * bag[color] / 100
    fold = round(obs / exp, 2)
    
    # Hypergeometric test for enrichment
    p = hypergeom.sf(obs - 1, M=100, n=bag[color], N=subset_size)
    
    sig = 'Yes' if p < 0.05 else 'No'
    print(color, "\t\t", obs, "\t", round(exp, 1), "\t", fold, "\t", round(p, 4), "\t", sig)

---

## 3. From Enrichment to Gene Expression

The colored balls illustrate the **logic** of enrichment analysis.
In genomics, the same question comes up constantly:

| Balls example | GO enrichment |
|---|---|
| 100 balls in a bag | All genes in the genome |
| Purple, red, blue... | GO categories (metabolism, signal transduction...) |
| Subset of 15 balls | Our differentially expressed gene list |
| "Is purple enriched?" | "Is signal transduction enriched?" |

Let's see how this works with real expression data.

### Load the data

In [ ]:
# Load expression data (samples in rows, genes in columns)
expression = pd.read_csv(REPO + '/data/bulk_rnaseq_counts.csv', index_col=0)
metadata = pd.read_csv(REPO + '/data/bulk_rnaseq_metadata.csv')

print("Expression:", expression.shape[0], "samples x", expression.shape[1], "genes")
print("Conditions:", list(metadata['condition'].unique()))
expression.head()

In [ ]:
metadata

### Test one gene

Let's pick a single gene and ask: does its expression differ between Normal and Treatment_A?

We use the same t-test from Day 3.

In [ ]:
# Boxplot of Gene_001 across conditions
gene = 'Gene_001'

plot_data = pd.DataFrame({
    'expression': expression[gene].values,
    'condition': metadata['condition'].values
})

sns.boxplot(data=plot_data, x='condition', y='expression', color='lightgray')
sns.stripplot(data=plot_data, x='condition', y='expression', size=8, color='black')
plt.title(gene)
plt.show()

In [ ]:
# T-test: Normal vs Treatment_A
normal_samples = metadata[metadata['condition'] == 'Normal']['sample_name'].values
treat_a_samples = metadata[metadata['condition'] == 'Treatment_A']['sample_name'].values

normal_vals = expression.loc[normal_samples, gene]
treat_a_vals = expression.loc[treat_a_samples, gene]

t_stat, p_value = stats.ttest_ind(normal_vals, treat_a_vals)

print(gene)
print("  Normal mean:     ", round(normal_vals.mean(), 1))
print("  Treatment_A mean:", round(treat_a_vals.mean(), 1))
print("  Fold change:     ", round(treat_a_vals.mean() / normal_vals.mean(), 2))
print("  P-value:         ", round(p_value, 6))
print()
print("This gene is upregulated in Treatment_A.")

### From one gene to many

In a real analysis, you would test **every gene** and correct for multiple testing.
Tools like [DESeq2](https://bioconductor.org/packages/DESeq2/) and [edgeR](https://bioconductor.org/packages/edgeR/) do this properly — you will see them in Week 2.

For now, **assume we ran a full analysis** and found these genes differentially expressed between Normal and Treatment_A:

In [ ]:
# Pre-computed DE results (from an analysis)
upregulated = ['Gene_001', 'Gene_002', 'Gene_003', 'Gene_004', 'Gene_005',
               'Gene_006', 'Gene_007', 'Gene_008', 'Gene_009', 'Gene_010',
               'Gene_011', 'Gene_012', 'Gene_013', 'Gene_014', 'Gene_015']

downregulated = ['Gene_016', 'Gene_017', 'Gene_018', 'Gene_019', 'Gene_020',
                 'Gene_021', 'Gene_022', 'Gene_023', 'Gene_024', 'Gene_025',
                 'Gene_026', 'Gene_027', 'Gene_028', 'Gene_029', 'Gene_030']

print("Upregulated genes:  ", len(upregulated))
print("Downregulated genes:", len(downregulated))
print()
print("These genes change significantly between Normal and Treatment_A.")
print("But what do they DO? Do they share a biological function?")

### GO enrichment

We have a table that tells us which genes belong to which GO (Gene Ontology) categories.
Let's check: are any GO terms **enriched** in our upregulated genes?

In [ ]:
# Load gene-to-GO mapping
go_table = pd.read_csv(REPO + '/data/go_annotations.csv')

print("GO annotations:", len(go_table), "rows")
print()
print("GO categories:")
print(go_table['go_name'].value_counts())
print()
go_table.head(10)

In [ ]:
# For each GO term: is it enriched in our upregulated genes?
total_genes = len(expression.columns)  # all genes in our dataset
list_size = len(upregulated)

print("Total genes in dataset:", total_genes)
print("Upregulated genes:", list_size)
print()

print("GO term".ljust(24), "In list".ljust(10), "In genome".ljust(12), "Fold".ljust(8), "P-value")
print("-" * 72)

for go_name in go_table['go_name'].unique():
    # Genes in this GO category (in the whole dataset)
    go_genes = go_table[go_table['go_name'] == go_name]['gene'].unique()
    go_size = len(go_genes)
    
    # How many of our upregulated genes are in this category?
    overlap = len(set(upregulated) & set(go_genes))
    
    # Expected and fold enrichment
    expected = list_size * go_size / total_genes
    fold = round(overlap / expected, 2) if expected > 0 else 0
    
    # Hypergeometric test (same as colored balls)
    p = hypergeom.sf(overlap - 1, M=total_genes, n=go_size, N=list_size)
    
    print(go_name.ljust(24), str(overlap).ljust(10), str(go_size).ljust(12), str(fold).ljust(8), round(p, 4))

### The multiple testing problem

We tested 5 GO categories. At a threshold of p < 0.05, we expect about 1 in 20 tests to look significant **purely by chance**. With thousands of GO terms (which is realistic), that means dozens of false positives.

**Bonferroni correction** is the simplest adjustment: divide the significance threshold by the number of tests.

$$\alpha_{\text{corrected}} = \frac{0.05}{\text{number of tests}}$$

It is actually strict — it can kill real effects — but it is easy to understand and guarantees that the chance of **any** false positive stays below 0.05. In real analyses with thousands of GO terms, less strict methods like **Benjamini-Hochberg** are more common because they allow more discoveries while still controlling the error rate.

In [ ]:
# Bonferroni correction
go_terms = go_table['go_name'].unique()
n_tests = len(go_terms)
bonferroni_threshold = 0.05 / n_tests

print("Number of tests:", n_tests)
print("Bonferroni threshold: 0.05 /", n_tests, "=", bonferroni_threshold)
print()

print("GO term".ljust(24), "P-value".ljust(12), "Significant?")
print("-" * 70)

for go_name in go_terms:
    go_genes = go_table[go_table['go_name'] == go_name]['gene'].unique()
    go_size = len(go_genes)
    overlap = len(set(upregulated) & set(go_genes))
    p = hypergeom.sf(overlap - 1, M=total_genes, n=go_size, N=list_size)
    
    if p < bonferroni_threshold:
        status = "YES (still after correction)"
    elif p < 0.05:
        status = "NO (not any more)"
    else:
        status = "NO"
    
    print(go_name.ljust(24), str(round(p, 6)).ljust(12), status)

### Interpretation

Signal transduction is enriched among the upregulated genes — and it survives Bonferroni correction. Metabolism was significant at p < 0.05 but does not survive the stricter threshold, so we would not report it as confidently.

In a real analysis, tools like [Enrichr](https://maayanlab.cloud/Enrichr/), [g:Profiler](https://biit.cs.ut.ee/gprofiler/), or [DAVID](https://david.ncifcrf.gov/) do this automatically across thousands of GO categories, with built-in multiple testing correction.

### 💡 Exercise 4.2: Downregulated Genes

**Task:** Repeat the GO enrichment for the **downregulated** genes.
Which GO term is enriched this time?

In [ ]:
# Your code here
# → replace 'upregulated' with 'downregulated' in the code above


---

## 4. Dimensionality Reduction

We just looked at individual genes. But what if we want to see the **overall pattern** across all genes at once?

With 100 genes, each sample is a point in 100-dimensional space — we cannot visualize that.

[**Principal Component Analysis (PCA)**](https://builtin.com/data-science/step-step-explanation-principal-component-analysis) is a technique used to simplify complex, high-dimensional data into fewer variables (principal components, PCs) while retaining most of the important information. It works by finding new, **uncorrelated axes** that maximize the spread (variance) of the data, making it easier to visualize **patterns** and capture the main **trends** in the data.

### PCA on a toy example

Let's start simple: 4 items described by 10 binary properties (1 = present, 0 = absent).

In [ ]:
# 4 items, 10 binary properties
toy = pd.DataFrame({
    'p1':  [1, 1, 0, 0],
    'p2':  [1, 1, 0, 0],
    'p3':  [1, 1, 0, 0],
    'p4':  [1, 1, 0, 0],
    'p5':  [1, 1, 0, 0],
    'p6':  [1, 0, 1, 0],
    'p7':  [1, 0, 1, 0],
    'p8':  [1, 0, 1, 0],
    'p9':  [1, 0, 0, 1],
    'p10': [1, 0, 0, 1],
}, index=['A', 'B', 'C', 'D'])

print("Each item has 10 properties. Is it obvious which items are similar?")
toy

There are **three trends** in this table:

1. **Trend 1 (p1–p5):** A and B share these 5 properties. C and D do not. *(5 properties)*
2. **Trend 2 (p6–p8):** A and C share these 3 properties. B and D do not. *(3 properties)*
3. **Trend 3 (p9–p10):** A and D share these 2 properties. B and C do not. *(2 properties)*

Each of them groups items differently. PCA aims to find these trends and **rank them by importance**.

In [ ]:
# Run PCA: reduce 10 properties to 3 principal components
pca = PCA(n_components=3)
coords = pca.fit_transform(toy)

# How much variance does each PC capture?
for i in range(3):
    pct = round(pca.explained_variance_ratio_[i] * 100, 1)
    print("PC" + str(i+1) + " explains", pct, "% of the variance")

print()
total = round(sum(pca.explained_variance_ratio_) * 100, 1)
print("Total:", total, "%")

The variance split is **50% / 30% / 20%** — directly proportional to the number of properties in each trend (5 / 3 / 2).

More properties backing a trend → more variance explained.

Let's plot PC1 vs PC2. Together they capture **80%** of the total variation — the two biggest trends.

In [ ]:
# Build results table for PC1 and PC2
toy_pca = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1]
}, index=toy.index)

# Plot
pc1_var = round(pca.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(pca.explained_variance_ratio_[1] * 100, 1)

sns.scatterplot(data=toy_pca, x='PC1', y='PC2', s=200)

# Add item labels
for name, row in toy_pca.iterrows():
    plt.text(row['PC1'] + 0.04, row['PC2'] + 0.04, name, fontsize=14, fontweight='bold')

# Axes through origin
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()

plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of toy data')
plt.show()

print("PC1 (50%) separates A, B from C, D — the biggest trend (5 properties).")
print("PC2 (30%) separates A, C from B, D — the second trend (3 properties).")
print("PC3 (20%) would separate A, D from B, C — but it's not shown here.")

### What did PCA just do?

We had 10 numbers per item — too many to visualize. PCA compressed them into 2 coordinates (PCs) while preserving **80%** of the information. The remaining 20% (trend 3, PC3) is lost in this 2D projection.

-> PCA finds the **main axes of variation** in your data and ranks them.

### PCA on gene expression data

Now the same idea on our 9 samples with 100 genes.
Two preparation steps are needed:
- **Log-transform** — raw counts are very skewed, log compresses the range
- **Scale** — make each gene have mean 0 and standard deviation 1, so no single gene dominates

In [ ]:
# Prepare the data
log_expression = np.log2(expression + 1)
X_scaled = StandardScaler().fit_transform(log_expression)

# Run PCA
pca_expr = PCA(n_components=5)
coords = pca_expr.fit_transform(X_scaled)

# Results
expr_pca = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'condition': metadata['condition'].values,
    'sample': metadata['sample_name'].values
})

# Plot
pc1_var = round(pca_expr.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(pca_expr.explained_variance_ratio_[1] * 100, 1)

sns.scatterplot(data=expr_pca, x='PC1', y='PC2', hue='condition', s=200)

plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()
plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of gene expression')
plt.show()

print("Samples from the same condition cluster together.")
print("The treatments are clearly separated from Normal.")

**Try it:** plot PC1 vs PC3 instead of PC1 vs PC2. Do the conditions still separate?

In [ ]:
# Try it here


### Scree plot: how many PCs matter?

In [ ]:
# Which PCs capture the most variation?
pc_labels = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
var_pct = [round(v * 100, 1) for v in pca_expr.explained_variance_ratio_]

sns.barplot(x=pc_labels, y=var_pct)
plt.ylabel('Variance explained (%)')
plt.title('Scree plot')
plt.show()

for i in range(len(pc_labels)):
    print(pc_labels[i] + ":", var_pct[i], "%")

### 💡 Exercise 4.3: PCA exploration

1. How much total variance do PC1 and PC2 together explain?
2. Try plotting PC1 vs PC3 — does it reveal anything new?

In [ ]:
# Your code here

# 1. Total variance of PC1 + PC2

# 2. PC1 vs PC3


---

## 5. Single-Cell Data: PCA vs UMAP

With only 9 bulk samples, PCA works well.
But what about datasets with **hundreds or thousands** of samples, like single-cell RNA-seq?

We have expression data for 200 individual cells from 4 immune cell types.
We will compare PCA and [**UMAP**](https://umap-learn.readthedocs.io/) on this dataset.

### Load single-cell data

In [ ]:
# Load single-cell expression (200 cells x 100 genes)
sc_expression = pd.read_csv(REPO + '/data/singlecell_expression.csv', index_col=0)
sc_metadata = pd.read_csv(REPO + '/data/singlecell_metadata.csv')

print("Expression:", sc_expression.shape[0], "cells x", sc_expression.shape[1], "genes")
print("Cell types:", dict(sc_metadata['cell_type'].value_counts()))

### PCA on single-cell data

In [ ]:
# Same steps: log-transform, scale, run PCA
sc_log = np.log2(sc_expression + 1)
sc_scaled = StandardScaler().fit_transform(sc_log)

sc_pca = PCA(n_components=2)
sc_pca_coords = sc_pca.fit_transform(sc_scaled)

sc_pca_df = pd.DataFrame({
    'PC1': sc_pca_coords[:, 0],
    'PC2': sc_pca_coords[:, 1],
    'cell_type': sc_metadata['cell_type'].values
})

pc1_var = round(sc_pca.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(sc_pca.explained_variance_ratio_[1] * 100, 1)

sns.scatterplot(data=sc_pca_df, x='PC1', y='PC2', hue='cell_type', s=30, alpha=0.7)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()
plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of single-cell data')
plt.show()

### UMAP on single-cell data

[UMAP](https://umap-learn.readthedocs.io/) (Uniform Manifold Approximation and Projection) is a **non-linear** method that focuses on preserving **local neighborhoods**: cells that are similar stay close together.

This makes it better at finding **clusters** in complex data.

In [ ]:
from umap import UMAP

# Run UMAP on the same scaled data
umap_model = UMAP(n_components=2, random_state=42)
umap_coords = umap_model.fit_transform(sc_scaled)

sc_umap_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'cell_type': sc_metadata['cell_type'].values
})

sns.scatterplot(data=sc_umap_df, x='UMAP1', y='UMAP2', hue='cell_type', s=30, alpha=0.7)
plt.title('UMAP of single-cell data')
plt.show()

print("UMAP tends to separate clusters more cleanly than PCA.")

**Try it:** re-run UMAP with `n_neighbors=5` instead of the default. What changes? Then try `n_neighbors=50`.

In [ ]:
# Try it here


### Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA panel
sns.scatterplot(data=sc_pca_df, x='PC1', y='PC2', hue='cell_type', s=30, alpha=0.7, ax=axes[0])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('PC1 (' + str(pc1_var) + '%)')
axes[0].set_ylabel('PC2 (' + str(pc2_var) + '%)')
axes[0].set_title('PCA')
for spine in ['top', 'right']:
    axes[0].spines[spine].set_visible(False)

# UMAP panel
sns.scatterplot(data=sc_umap_df, x='UMAP1', y='UMAP2', hue='cell_type', s=30, alpha=0.7, ax=axes[1])
axes[1].set_title('UMAP')

plt.tight_layout()
plt.show()

### When to use each?

| | PCA | UMAP |
|---|---|---|
| Best for | Understanding which trends dominate | Visualizing clusters |
| Type | Linear | Non-linear |
| Distances | Meaningful — far apart on the plot means far apart in the data | Not meaningful — only neighborhoods are preserved |
| Interpretable? | Yes (variance explained, loadings) | Less so |
| Typical use | Bulk RNA-seq, quality control | Single-cell RNA-seq |
| Links | [Wikipedia](https://en.wikipedia.org/wiki/Principal_component_analysis) | [UMAP docs](https://umap-learn.readthedocs.io/) |

**Linear vs non-linear:** PCA finds straight-line trends. If two samples differ a lot in gene expression, they will be far apart on the PCA plot — distances are proportional to real differences. UMAP, on the other hand, focuses on preserving which samples are *neighbors*. It can pull apart clusters very clearly, but the distance between two clusters on a UMAP plot does not tell you how different they really are.

---

## Summary

### Enrichment
- **Enrichment = observed vs expected** in a subset
- **Permutation test** — simulate random subsets to estimate significance
- **Hypergeometric test** — exact calculation
- Same logic applies to colored balls and to GO terms
- **Multiple testing correction** (Bonferroni) — stricter threshold when testing many categories

### Differential expression
- For each gene: is it significantly different between conditions?
- Then ask: do the DE genes share a biological function? → **GO enrichment**
- Tools like DESeq2 and edgeR handle the full analysis (Week 2)

### Dimensionality reduction
- **PCA** finds the main **trends** — linear, interpretable
- **UMAP** preserves **neighborhoods** — non-linear, better for clusters
- **Log-transform** and **scale** expression data before PCA

### Next week

- **Day 5:** Bulk RNA-seq — full differential expression pipeline
- **Day 6:** Single-cell RNA-seq — from raw data to cell types
- **Day 7:** ATAC-seq — chromatin accessibility
- **Day 8:** Bring your data
